In [28]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import PowerTransformer

In [29]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [30]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [31]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [32]:
# Check null values in D3 
clinical_train.isnull().sum().sum()

0

### COX assumption in Train data

In [33]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.00 0.97      0.05
LBP_003_PET                                                      0.00 0.97      0.05
LBP_012_CT                                                       0.00 0.96      0.06
LBP_012_PET                                                      0.01 0.94      0.09
LBP_021_CT                                                       0.00 0.98      0.04
LBP_021_PET                                                      0.01 0.94      0.10
LBP_030_CT                                                       0.00 0.99      0.02
LBP_030_PET       

In [34]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


In [35]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.00 0.95      0.07
LBP_003_PET                                                      0.08 0.77      0.37
LBP_012_CT                                                       0.00 0.95      0.07
LBP_012_PET                                                      0.18 0.67      0.57
LBP_021_CT                                                       0.15 0.69      0.53
LBP_021_PET                                                      0.01 0.94      0.09
LBP_030_CT                                                       0.00 0.99      0.01
LBP_030_PET       

In [36]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


## Test dataset: MAASTRO 

In [37]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [38]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [39]:
# Need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [40]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [41]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [42]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [43]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# Set y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [44]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [45]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [46]:
# Change the name of a column 'OS_event' in the clincial_test 
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [47]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

# Yeo-Johnson Transformation

In [48]:
# Copy the original X for later 
original_X = X.copy()

In [49]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']
### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
pt = PowerTransformer(method='yeo-johnson')
X_numeric_std = pt.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [52]:
# Standardize X_MAASTRO 
MAASTRO_new = X_MAASTRO.copy()
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [53]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = MAASTRO_new_std 

# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [54]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 03:28:22,068] A new study created in memory with name: no-name-ea79c532-3e9d-4eb0-81ee-e4b51823644d
python(7418) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/1 [00:00<?, ?it/s]

[W 2024-04-18 03:28:29,547] Trial 0 failed with parameters: {} because of the following error: LinAlgError('Matrix is singular.').
Traceback (most recent call last):
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/64/jkqp6xyx2hj50dmd2pqfm3780000gn/T/ipykernel_6391/487959643.py", line 62, in objective
    model.fit(X_train_std, y_train)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/sksurv/linear_model/coxph.py", line 449, in fit
    delta = solve(
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 220, in solve
    _solve_check(n, info)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 29, in _solve_check
    raise LinAlgError('Matrix is singular.')
numpy.linalg.LinAlgError: Matrix is singular.
[W 2024-04-18 03:28:29,555] Trial 0 faile

LinAlgError: Matrix is singular.

In [55]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

In [56]:
# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

ValueError: No trials are completed yet.

In [57]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

ValueError: No trials are completed yet.

#### Test

In [58]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [59]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

ValueError: search direction contains NaN or infinite values

In [60]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [61]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

NameError: name 'c_index' is not defined

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [62]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 03:28:52,381] A new study created in memory with name: no-name-5f7a4320-5f1e-4d08-a9d1-32398e006000


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7276785714285714
Fold 3 C-index: 0.7867647058823529
Fold 4 C-index: 0.689873417721519


[I 2024-04-18 03:29:30,738] A new study created in memory with name: no-name-858acf81-ccd6-4195-ba3a-34cecfbdd29c


Fold 5 C-index: 0.7746478873239436
[I 2024-04-18 03:29:30,698] Trial 0 finished with value: 0.7161392368175977 and parameters: {}. Best is trial 0 with value: 0.7161392368175977.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7161392368175977], datetime_start=datetime.datetime(2024, 4, 18, 3, 28, 52, 565389), datetime_complete=datetime.datetime(2024, 4, 18, 3, 29, 30, 697857), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7161392368175977


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651724235042
Fold 2 IBS: 0.2215779146315195
Fold 3 IBS: 0.20453594072931716
Fold 4 IBS: 0.22473803872764997
Fold 5 IBS: 0.21812431098174057
[I 2024-04-18 03:30:04,788] Trial 0 finished with value: 0.2165905444625155 and parameters: {}. Best is trial 0 with value: 0.2165905444625155.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2165905444625155], datetime_start=datetime.datetime(2024, 4, 18, 3, 29, 30, 935602), datetime_complete=datetime.datetime(2024, 4, 18, 3, 30, 4, 787702), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2165905444625155


In [63]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [64]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.716
train_ibs:  0.217


#### Test

In [65]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [66]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.573


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [67]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [68]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 03:30:05,514] A new study created in memory with name: no-name-915df40a-2c58-42d4-92a4-df19be137f77


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7922077922077922
Fold 2 C-index: 0.6651785714285714
Fold 3 C-index: 0.6715686274509803
Fold 4 C-index: 0.6075949367088608


[I 2024-04-18 03:30:46,500] A new study created in memory with name: no-name-e3938e04-ec70-4359-bea9-437716aa158b


Fold 5 C-index: 0.6525821596244131
[I 2024-04-18 03:30:46,408] Trial 0 finished with value: 0.6778264174841235 and parameters: {}. Best is trial 0 with value: 0.6778264174841235.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6778264174841235], datetime_start=datetime.datetime(2024, 4, 18, 3, 30, 5, 617174), datetime_complete=datetime.datetime(2024, 4, 18, 3, 30, 46, 407987), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6778264174841235


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21533544230916105
Fold 2 IBS: 0.35684861315058924
Fold 3 IBS: 0.2999824518344091
Fold 4 IBS: 0.3450292483985389
Fold 5 IBS: 0.34672666404044594
[I 2024-04-18 03:31:32,397] Trial 0 finished with value: 0.31278448394662883 and parameters: {}. Best is trial 0 with value: 0.31278448394662883.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.31278448394662883], datetime_start=datetime.datetime(2024, 4, 18, 3, 30, 46, 659703), datetime_complete=datetime.datetime(2024, 4, 18, 3, 31, 32, 397126), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.31278448394662883


In [69]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [70]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.678
train_ibs:  0.313


#### Test

In [71]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [72]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.55


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.419


In [73]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [74]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 03:31:36,622] A new study created in memory with name: no-name-986e34a2-9863-4add-8735-1f1783c7d63d


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7835497835497836
Fold 2 C-index: 0.6875
Fold 3 C-index: 0.6617647058823529
Fold 4 C-index: 0.6497890295358649
Fold 5 C-index: 0.6338028169014085
[I 2024-04-18 03:32:18,521] Trial 0 finished with value: 0.6832812671738819 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6832812671738819.
Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.6964285714285714
Fold 3 C-index: 0.7009803921568627
Fold 4 C-index: 0.679324894514768
Fold 5 C-index: 0.676056338028169
[I 2024-04-18 03:32:55,899] Trial 1 finished with value: 0.70640219506983 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.70640219506983.
Fold 1 C-index: 0.7705627705627706
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7058823529411765
Fold 4 C-index: 0.6877637130801688
Fold 5 C-index: 0.6807511737089202
[I 2024-04-18 03:33:34,772] Trial 2 finished with value: 0.710956287772893 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 wi

Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.7652582159624414
[I 2024-04-18 03:45:37,647] Trial 24 finished with value: 0.7230519534431709 and parameters: {'l1_ratio': 0.004279993897261608}. Best is trial 12 with value: 0.7499920028304033.
Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.7142857142857143
Fold 3 C-index: 0.7205882352941176
Fold 4 C-index: 0.7046413502109705
Fold 5 C-index: 0.6854460093896714
[I 2024-04-18 03:46:10,875] Trial 25 finished with value: 0.718239015082848 and parameters: {'l1_ratio': 0.17401480064872898}. Best is trial 12 with value: 0.7499920028304033.
Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.71875
Fold 3 C-index: 0.7401960784313726
Fold 4 C-index: 0.7172995780590717
Fold 5 C-index: 0.7183098591549296
[I 2024-04-18 03:46:42,550] Trial 26 finished with value: 0.7321578563758281 and parameters: {'l1_ratio': 0.08693050661949489}. Best

Fold 1 C-index: 0.7748917748917749
Fold 2 C-index: 0.6875
Fold 3 C-index: 0.6666666666666666
Fold 4 C-index: 0.6497890295358649
Fold 5 C-index: 0.6384976525821596
[I 2024-04-18 03:58:08,516] Trial 48 finished with value: 0.6834690247352933 and parameters: {'l1_ratio': 0.5683872043393509}. Best is trial 12 with value: 0.7499920028304033.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7142857142857143
Fold 3 C-index: 0.7254901960784313
Fold 4 C-index: 0.7172995780590717
Fold 5 C-index: 0.7089201877934272
[I 2024-04-18 03:58:35,712] Trial 49 finished with value: 0.7255800876242814 and parameters: {'l1_ratio': 0.12391166716004895}. Best is trial 12 with value: 0.7499920028304033.
Fold 1 C-index: 0.7705627705627706
Fold 2 C-index: 0.7142857142857143
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.6807511737089202
[I 2024-04-18 03:59:06,771] Trial 50 finished with value: 0.7146538110860157 and parameters: {'l1_ratio': 0.19334719885624377}. Best i

...

Fold 1 IBS: 0.18794653865421088
Fold 2 IBS: 0.26160300244411944
Fold 3 IBS: 0.28499323418158673
Fold 4 IBS: 0.37889222817047347
Fold 5 IBS: 0.33256810688981997
[I 2024-04-18 04:20:33,833] Trial 0 finished with value: 0.2892006220680421 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.2892006220680421.
Fold 1 IBS: 0.1851004644537256
Fold 2 IBS: 0.23181203791678803
Fold 3 IBS: 0.25749421908055464
Fold 4 IBS: 0.2982983305423583
Fold 5 IBS: 0.31018586605458565
[I 2024-04-18 04:21:00,967] Trial 1 finished with value: 0.2565781836096025 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.2565781836096025.
Fold 1 IBS: 0.18500316807064598
Fold 2 IBS: 0.22643389657788915
Fold 3 IBS: 0.24787833814277196
Fold 4 IBS: 0.27820420873175505
Fold 5 IBS: 0.30188670644796195
[I 2024-04-18 04:21:27,099] Trial 2 finished with value: 0.24788126359420484 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.247881263594204

Fold 1 IBS: 0.1886783645535632
Fold 2 IBS: 0.21041644126736042
Fold 3 IBS: 0.2054630322886
Fold 4 IBS: 0.23163040552306483
Fold 5 IBS: 0.2666400705889357
[I 2024-04-18 04:31:19,333] Trial 25 finished with value: 0.22056566284430482 and parameters: {'l1_ratio': 0.0861486623508641}. Best is trial 12 with value: 0.21025006071733993.
Fold 1 IBS: 0.18511999812868693
Fold 2 IBS: 0.22725446344072614
Fold 3 IBS: 0.24922821097430353
Fold 4 IBS: 0.2809228106336443
Fold 5 IBS: 0.30324147396616297
[I 2024-04-18 04:31:44,593] Trial 26 finished with value: 0.24915339142870474 and parameters: {'l1_ratio': 0.23521758130887002}. Best is trial 12 with value: 0.21025006071733993.
Fold 1 IBS: 0.1997523133624926
Fold 2 IBS: 0.29601537826867064
Fold 3 IBS: 0.2910628698762797
Fold 4 IBS: 0.3814530039898754
Fold 5 IBS: 0.340768470315745
[I 2024-04-18 04:32:12,023] Trial 27 finished with value: 0.30181040716261265 and parameters: {'l1_ratio': 0.8345729974660353}. Best is trial 12 with value: 0.2102500607173399

Fold 1 IBS: 0.19680688470972987
Fold 2 IBS: 0.22059349240521664
Fold 3 IBS: 0.17790674695647057
Fold 4 IBS: 0.22374810517623409
Fold 5 IBS: 0.24705340079146137
[I 2024-04-18 04:41:49,911] Trial 50 finished with value: 0.2132217260078225 and parameters: {'l1_ratio': 0.04088364878367627}. Best is trial 12 with value: 0.21025006071733993.
Fold 1 IBS: 0.19755520757290046
Fold 2 IBS: 0.22065105806317759
Fold 3 IBS: 0.17600353880473255
Fold 4 IBS: 0.22380275595019244
Fold 5 IBS: 0.24576191296373862
[I 2024-04-18 04:42:14,527] Trial 51 finished with value: 0.21275489467094832 and parameters: {'l1_ratio': 0.03840625231138948}. Best is trial 12 with value: 0.21025006071733993.
Fold 1 IBS: 0.1876867489201972
Fold 2 IBS: 0.21276180085856197
Fold 3 IBS: 0.21328703368625607
Fold 4 IBS: 0.23657773218133069
Fold 5 IBS: 0.2727641534627712
[I 2024-04-18 04:42:39,426] Trial 52 finished with value: 0.22461549382182344 and parameters: {'l1_ratio': 0.1036080990209961}. Best is trial 12 with value: 0.210250

Fold 5 IBS: 0.21802220001630135
[I 2024-04-18 04:51:47,280] Trial 74 finished with value: 0.21649156134151465 and parameters: {'l1_ratio': 0.0024082397705825313}. Best is trial 68 with value: 0.2098349784451214.
Fold 1 IBS: 0.18713776288010345
Fold 2 IBS: 0.21415584059731327
Fold 3 IBS: 0.21897376815099812
Fold 4 IBS: 0.24055905612506784
Fold 5 IBS: 0.277284314188442
[I 2024-04-18 04:52:12,505] Trial 75 finished with value: 0.22762214838838496 and parameters: {'l1_ratio': 0.1176051017445135}. Best is trial 68 with value: 0.2098349784451214.
Fold 1 IBS: 0.1891424886558718
Fold 2 IBS: 0.2090637464903817
Fold 3 IBS: 0.20273246646212909
Fold 4 IBS: 0.2229542737140009
Fold 5 IBS: 0.2644368439058008
[I 2024-04-18 04:52:37,187] Trial 76 finished with value: 0.21766596384563686 and parameters: {'l1_ratio': 0.08047320826246168}. Best is trial 68 with value: 0.2098349784451214.
Fold 1 IBS: 0.20279705638688392
Fold 2 IBS: 0.2210376400175663
Fold 3 IBS: 0.20329448071318942
Fold 4 IBS: 0.2241837848

Fold 1 IBS: 0.1886593576885953
Fold 2 IBS: 0.21050054296640958
Fold 3 IBS: 0.2056396345482091
Fold 4 IBS: 0.2317394182543497
Fold 5 IBS: 0.26677120778113017
[I 2024-04-18 05:02:02,591] Trial 99 finished with value: 0.22066203224773878 and parameters: {'l1_ratio': 0.08650676983934198}. Best is trial 92 with value: 0.20967239285768788.


* Best trial for IBS: 
 FrozenTrial(number=92, state=TrialState.COMPLETE, values=[0.20967239285768788], datetime_start=datetime.datetime(2024, 4, 18, 4, 58, 45, 541205), datetime_complete=datetime.datetime(2024, 4, 18, 4, 59, 10, 544216), params={'l1_ratio': 0.023103419286485582}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=92, value=None)


* Best Score for IBS: 
 0.20967239285768788


In [75]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [76]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.753
train_ibs:  0.21


#### Test

In [77]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [78]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.022395157685824067)

test_cindex : 0.573


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.023103419286485582)

test_ibs:  0.221


In [79]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [80]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 05:02:03,014] A new study created in memory with name: no-name-b0b90f8f-ec6b-49f3-b385-873987dd70ca


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7745098039215687
Fold 4 C-index: 0.7383966244725738
Fold 5 C-index: 0.7230046948356808
[I 2024-04-18 05:03:19,740] Trial 0 finished with value: 0.7366032203169602 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7366032203169602.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.7721518987341772
Fold 5 C-index: 0.6854460093896714
[I 2024-04-18 05:03:33,123] Trial 1 finished with value: 0.7181620567966567 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 1 C-index: 0.6038961038961039
Fold 2 C-index: 0.7165178571428571
Fold 3 C-index: 0.7009803921568627
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.6032863849765259
[I 2024-04-18 05:09:18,586] Trial 16 finished with value: 0.6633327721070438 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 7, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.36871324569404207, 'max_features': None, 'min_weight_fraction_leaf': 0.1048543527585735, 'warm_start': False}. Best is trial 12 with value: 0.7438602547617209.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 05:09:34,901] Trial 17 finished with value: 0.5 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 1, 'max_depth': 15, 'n_estimators': 417, 'oob_score': True, 'max_samples': 0.1424705672746025, 'max_features': None, 'min_weight_fraction_leaf': 0.20205510510599584, 'warm_start': F

Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.812206572769953
[I 2024-04-18 05:12:58,350] Trial 31 finished with value: 0.7936613975238874 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 8, 'min_samples_leaf': 12, 'max_depth': 6, 'n_estimators': 215, 'oob_score': True, 'max_samples': 0.7258118850709611, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.020186907613437947, 'warm_start': True}. Best is trial 26 with value: 0.7952997362091728.
Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.8616071428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.8169014084507042
[I 2024-04-18 05:13:01,681] Trial 32 finished with value: 0.7892741864327053 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 8, 'min_samples_leaf': 11, 'max_depth': 8, 'n_estimators': 149, 'oob_score': True, 'max_samples': 0.7060378252364139, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.0461866753469

...

Fold 1 IBS: 0.20444739845582136
Fold 2 IBS: 0.17446698341767913
Fold 3 IBS: 0.1910994493903993
Fold 4 IBS: 0.1668825886510828
Fold 5 IBS: 0.21103427734544067
[I 2024-04-18 05:37:03,065] Trial 32 finished with value: 0.18958613945208463 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 7, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 279, 'oob_score': False, 'max_samples': 0.6325115094238634, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.0760629688237641}. Best is trial 21 with value: 0.18758259521522488.
Fold 1 IBS: 0.21394095082970305
Fold 2 IBS: 0.18957239420831615
Fold 3 IBS: 0.18237897227425182
Fold 4 IBS: 0.17846689832832333
Fold 5 IBS: 0.2185435449686763
[I 2024-04-18 05:37:04,707] Trial 33 finished with value: 0.19658055212185413 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 19, 'oob_score': False, 'max_samples': 0.7088178154422675, 'max_features': 'auto', 'min_weight_fraction_leaf':

Fold 1 IBS: 0.21397545241919333
Fold 2 IBS: 0.22060585341756236
Fold 3 IBS: 0.20519466411613427
Fold 4 IBS: 0.2248054198587066
Fold 5 IBS: 0.21758241895615993
[I 2024-04-18 05:40:46,939] Trial 48 finished with value: 0.2164327617535513 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 4, 'max_depth': 1, 'n_estimators': 324, 'oob_score': True, 'max_samples': 0.1840472657347053, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.44962767788541247}. Best is trial 39 with value: 0.18741882102325463.
Fold 1 IBS: 0.20733215663048163
Fold 2 IBS: 0.19840873877308707
Fold 3 IBS: 0.1896647344947966
Fold 4 IBS: 0.2057146078926162
Fold 5 IBS: 0.2152447795126918
[I 2024-04-18 05:40:54,545] Trial 49 finished with value: 0.20327300346073468 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 17, 'min_samples_leaf': 20, 'max_depth': 4, 'n_estimators': 187, 'oob_score': True, 'max_samples': 0.4611544071310423, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.18697650055709675
Fold 2 IBS: 0.18964751614322145
Fold 3 IBS: 0.17157067951262514
Fold 4 IBS: 0.17048079186700552
Fold 5 IBS: 0.20812563312237994
[I 2024-04-18 05:51:31,738] Trial 64 finished with value: 0.18536022424046578 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 381, 'oob_score': True, 'max_samples': 0.36087148528744895, 'max_features': None, 'min_weight_fraction_leaf': 0.10501492814170305}. Best is trial 64 with value: 0.18536022424046578.
Fold 1 IBS: 0.185980269157479
Fold 2 IBS: 0.18929665176523391
Fold 3 IBS: 0.17261558628019577
Fold 4 IBS: 0.17528863593015434
Fold 5 IBS: 0.20987976574068176
[I 2024-04-18 05:52:20,295] Trial 65 finished with value: 0.18661218177474898 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 389, 'oob_score': False, 'max_samples': 0.35503340896452096, 'max_features': None, 'min_weight_fraction_leaf':

Fold 1 IBS: 0.2139852535910293
Fold 2 IBS: 0.22079135086710738
Fold 3 IBS: 0.2049571432154721
Fold 4 IBS: 0.22449984942587378
Fold 5 IBS: 0.21784420818016068
[I 2024-04-18 05:59:40,563] Trial 80 finished with value: 0.21641556105592863 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 3, 'n_estimators': 401, 'oob_score': False, 'max_samples': 0.23317876663764808, 'max_features': None, 'min_weight_fraction_leaf': 0.13195032778351135}. Best is trial 64 with value: 0.18536022424046578.
Fold 1 IBS: 0.18698963922132994
Fold 2 IBS: 0.19117908340974094
Fold 3 IBS: 0.1716425830420473
Fold 4 IBS: 0.1706227695034198
Fold 5 IBS: 0.20890166136319582
[I 2024-04-18 06:00:30,437] Trial 81 finished with value: 0.1858671473079468 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 7, 'min_samples_leaf': 3, 'max_depth': 3, 'n_estimators': 342, 'oob_score': False, 'max_samples': 0.3619448678095568, 'max_features': None, 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.19987889711029083
Fold 2 IBS: 0.18568352389906653
Fold 3 IBS: 0.18380646487398802
Fold 4 IBS: 0.16033813976360883
Fold 5 IBS: 0.20642365546571148
[I 2024-04-18 06:14:02,779] Trial 96 finished with value: 0.18722613622253315 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 6, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 363, 'oob_score': False, 'max_samples': 0.4255182801365352, 'max_features': None, 'min_weight_fraction_leaf': 0.08056036420837291}. Best is trial 84 with value: 0.1832464308635407.
Fold 1 IBS: 0.19365837237754394
Fold 2 IBS: 0.183435405665406
Fold 3 IBS: 0.18008910470800357
Fold 4 IBS: 0.17339462295545943
Fold 5 IBS: 0.20781358095494984
[I 2024-04-18 06:14:52,156] Trial 97 finished with value: 0.18767821733227258 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 1, 'n_estimators': 344, 'oob_score': False, 'max_samples': 0.4693979321374093, 'max_features': None, 'min_weight_fraction_leaf': 0

In [81]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [82]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.864
train_ibs:  0.183


#### Test

In [83]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

In [84]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=11, max_features='auto', max_leaf_nodes=15,
                     max_samples=0.9847467855272036, min_samples_leaf=4,
                     min_samples_split=7,
                     min_weight_fraction_leaf=0.04749646122417352,
                     n_estimators=230, random_state=123, warm_start=True)

test_cindex:  0.629


RandomSurvivalForest(max_depth=2, max_features=None, max_leaf_nodes=7,
                     max_samples=0.3902292236281065, min_samples_leaf=4,
                     min_samples_split=5,
                     min_weight_fraction_leaf=0.0959966989881746,
                     n_estimators=344, random_state=123)

test_ibs:  0.205


In [85]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [86]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [87]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 06:15:59,301] A new study created in memory with name: no-name-c81615ea-f86c-43b7-bba2-b91156804215


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6623376623376623
Fold 2 C-index: 0.8616071428571429
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.7605633802816901
[I 2024-04-18 06:16:01,892] Trial 0 finished with value: 0.7784578545189774 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7784578545189774.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 06:16:08,988] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.8526785714285714
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8776371308016878
Fold 5 C-index: 0.7323943661971831
[I 2024-04-18 06:17:18,505] Trial 16 finished with value: 0.7903520467896392 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 13, 'max_depth': 10, 'n_estimators': 95, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.884183184043622, 'min_weight_fraction_leaf': 0.10581506507448754}. Best is trial 3 with value: 0.802961090796227.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 06:17:20,671] Trial 17 finished with value: 0.5 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 4, 'n_estimators': 226, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.37217752523657055, 'min_weight_fraction_leaf': 0.1994767874105

Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8860759493670886
Fold 5 C-index: 0.7746478873239436
[I 2024-04-18 06:17:49,282] Trial 31 finished with value: 0.8023701302106281 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 10, 'max_depth': 3, 'n_estimators': 209, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.9175308644810205, 'min_weight_fraction_leaf': 0.1088935662786611}. Best is trial 3 with value: 0.802961090796227.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8776371308016878
Fold 5 C-index: 0.7652582159624414
[I 2024-04-18 06:17:51,384] Trial 32 finished with value: 0.7950579336258078 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 6, 'min_samples_leaf': 11, 'max_depth': 3, 'n_estimators': 211, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.9467713628420997, 'min_weight_fraction_leaf': 0.

...

Fold 1 IBS: 0.18973195855840097
Fold 2 IBS: 0.19901752012782084
Fold 3 IBS: 0.1553163395473765
Fold 4 IBS: 0.13719070606611738
Fold 5 IBS: 0.22779776404493374
[I 2024-04-18 06:34:49,020] Trial 30 finished with value: 0.18181085766892985 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 305, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.8220971907615917, 'min_weight_fraction_leaf': 0.076444606879153}. Best is trial 28 with value: 0.18105604370316586.
Fold 1 IBS: 0.2164877455822723
Fold 2 IBS: 0.18827035681808446
Fold 3 IBS: 0.16565039781780372
Fold 4 IBS: 0.1417428766421557
Fold 5 IBS: 0.21575647924950747
[I 2024-04-18 06:35:22,842] Trial 31 finished with value: 0.18558157122196473 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 12, 'n_estimators': 394, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.91

Fold 1 IBS: 0.21196016007242774
Fold 2 IBS: 0.21737810285668177
Fold 3 IBS: 0.20221519268515586
Fold 4 IBS: 0.21985834792032022
Fold 5 IBS: 0.21707100228614096
[I 2024-04-18 06:38:38,747] Trial 45 finished with value: 0.21369656116414532 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 11, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 226, 'oob_score': False, 'warm_start': False, 'max_features': 1, 'max_samples': 0.5998350649536632, 'min_weight_fraction_leaf': 0.05464331917294846}. Best is trial 33 with value: 0.18058942699185546.
Fold 1 IBS: 0.18076053104236317
Fold 2 IBS: 0.2172100382529918
Fold 3 IBS: 0.15472808053026357
Fold 4 IBS: 0.14511587127121353
Fold 5 IBS: 0.2339666086051307
[I 2024-04-18 06:38:47,434] Trial 46 finished with value: 0.18635622594039253 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 13, 'min_samples_leaf': 2, 'max_depth': 5, 'n_estimators': 185, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.840

Fold 1 IBS: 0.1940353275621495
Fold 2 IBS: 0.19268012360689624
Fold 3 IBS: 0.18110607070067974
Fold 4 IBS: 0.18845548212963192
Fold 5 IBS: 0.2126254989546211
[I 2024-04-18 06:42:56,849] Trial 60 finished with value: 0.1937805005907957 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 14, 'min_samples_leaf': 12, 'max_depth': 12, 'n_estimators': 305, 'oob_score': True, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.8575527275336559, 'min_weight_fraction_leaf': 0.18982981109623043}. Best is trial 33 with value: 0.18058942699185546.
Fold 1 IBS: 0.1881809892682945
Fold 2 IBS: 0.19390719781855054
Fold 3 IBS: 0.1588739037458674
Fold 4 IBS: 0.1367801254919651
Fold 5 IBS: 0.2266847482237021
[I 2024-04-18 06:43:16,674] Trial 61 finished with value: 0.18088539290967592 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 19, 'n_estimators': 362, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.7615

Fold 1 IBS: 0.20438643925191893
Fold 2 IBS: 0.1844312624795873
Fold 3 IBS: 0.17002168747641047
Fold 4 IBS: 0.14791525733420058
Fold 5 IBS: 0.21362624602582111
[I 2024-04-18 06:47:08,906] Trial 75 finished with value: 0.18407617851358768 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 11, 'n_estimators': 296, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.6651148142523113, 'min_weight_fraction_leaf': 0.0005666053087619444}. Best is trial 74 with value: 0.18029547957640585.
Fold 1 IBS: 0.19306514020322818
Fold 2 IBS: 0.1842925536567217
Fold 3 IBS: 0.16281493341095712
Fold 4 IBS: 0.15026196851923793
Fold 5 IBS: 0.20907571723485416
[I 2024-04-18 06:47:20,125] Trial 76 finished with value: 0.1799020626049998 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 273, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.6121

Fold 1 IBS: 0.19250257566167933
Fold 2 IBS: 0.18261531719008467
Fold 3 IBS: 0.16180974496548703
Fold 4 IBS: 0.14890218044290915
Fold 5 IBS: 0.2130135609453814
[I 2024-04-18 06:49:56,072] Trial 90 finished with value: 0.1797686758411083 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 4, 'max_depth': 9, 'n_estimators': 202, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.5935532822743834, 'min_weight_fraction_leaf': 0.007362099342399538}. Best is trial 88 with value: 0.1785581840441121.
Fold 1 IBS: 0.19466401388017032
Fold 2 IBS: 0.18486336673650017
Fold 3 IBS: 0.16588446237479346
Fold 4 IBS: 0.1465134541216928
Fold 5 IBS: 0.21287370502296324
[I 2024-04-18 06:50:04,524] Trial 91 finished with value: 0.18095980042722398 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 197, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.5826023

In [88]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [89]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.886
train_ibs:  0.179


#### Test

In [90]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [91]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=9, max_features=None, max_leaf_nodes=14,
                   max_samples=0.965941994806725, min_samples_split=7,
                   min_weight_fraction_leaf=0.017267854864073284,
                   n_estimators=336, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.662


ExtraSurvivalTrees(max_depth=9, max_features=0.1, max_leaf_nodes=20,
                   max_samples=0.5945245081220258, min_samples_leaf=4,
                   min_samples_split=12,
                   min_weight_fraction_leaf=0.006841645987133443,
                   n_estimators=279, oob_score=True, random_state=123,
                   warm_start=True)

IBS: 0.2


In [92]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [93]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 06:51:27,493] A new study created in memory with name: no-name-1b2a5feb-3b43-4509-918d-fec944c9fb4c


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 06:52:49,991] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 06:53:28,466] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 07:15:49,912] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7375077219334873.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 07:18:22,252] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 07:47:26,575] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 22 with value: 0.7577518089481858.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5654008438818565
Fold 5 C-index: 0.5821596244131455
[I 2024-04-18 07:50:19,398] Trial 26 finished with value: 0.5295120936590003 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632,

...

Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-18 09:53:08,815] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.36065535241543395, 'n_estimators': 435, 'criterion': 'squared_error', 'ccp_alpha': 1.5126136571072866, 'min_weight_fraction_leaf': 0.2002957617254777, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.8814649292460881, 'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2}. Best is trial 41 with value: 0.21552892877484067.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-18 09:53:39,524] Trial 56 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.2818850250060197, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.23

Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-18 10:00:09,872] Trial 66 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9998769833869746, 'learning_rate': 0.003967598379054899, 'dropout_rate': 0.17654958266634313, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 0.8981368148483696, 'min_weight_fraction_leaf': 0.03518122515344503, 'max_features': 'auto', 'min_impurity_decrease': 7.140027633149786e-05, 'validation_fraction': 0.6362228335648394, 'min_samples_split': 17, 'max_leaf_nodes': 15, 'min_samples_leaf': 18, 'max_depth': 1}. Best is trial 41 with value: 0.21552892877484067.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-18 10:00:47,243] Trial 67 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.8945386029541793, 'learning_rate': 0.01494075298406419, 'dropout_rate': 0.20

Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-18 10:08:01,015] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.7865219593038091, 'learning_rate': 0.05776064499915696, 'dropout_rate': 0.25073094184082545, 'n_estimators': 410, 'criterion': 'squared_error', 'ccp_alpha': 1.2846220433570537, 'min_weight_fraction_leaf': 0.32348748582358233, 'max_features': 1, 'min_impurity_decrease': 2.1196884133822008e-06, 'validation_fraction': 0.854746843935587, 'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 11, 'max_depth': 3}. Best is trial 75 with value: 0.20755403023595909.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-18 10:08:23,611] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.74450491092411, 'learning_rate': 0.08426314282635561,

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-18 10:15:51,520] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9047974654661481, 'learning_rate': 0.053489441535809173, 'dropout_rate': 0.15111205819257206, 'n_estimators': 351, 'criterion': 'squared_error', 'ccp_alpha': 0.8515398041309932, 'min_weight_fraction_leaf': 0.21790404115359022, 'max_features': 'auto', 'min_impurity_decrease': 2.7145979471070252e-06, 'validation_fraction': 0.8752408074269059, 'min_samples_split': 20, 'max_leaf_nodes': 16, 'min_samples_leaf': 13, 'max_depth': 1}. Best is trial 75 with value: 0.20755403023595909.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-18 10:16:35,441] Trial 89 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.8838515048108662, 'learning_rate': 0.011504505

Fold 3 IBS: 0.20030580413616672
Fold 4 IBS: 0.21827950204883104
Fold 5 IBS: 0.2160767264619057
[I 2024-04-18 10:24:43,673] Trial 99 finished with value: 0.21234381967629634 and parameters: {'subsample': 0.7984456569855936, 'learning_rate': 0.07921297533010697, 'dropout_rate': 0.36361374787631456, 'n_estimators': 498, 'criterion': 'squared_error', 'ccp_alpha': 0.004455045338579978, 'min_weight_fraction_leaf': 0.23447872920034368, 'max_features': 0.1, 'min_impurity_decrease': 0.00015837847703956763, 'validation_fraction': 0.4378698142708704, 'min_samples_split': 20, 'max_leaf_nodes': 20, 'min_samples_leaf': 15, 'max_depth': 5}. Best is trial 75 with value: 0.20755403023595909.


* Best trial for IBS: 
 FrozenTrial(number=75, state=TrialState.COMPLETE, values=[0.20755403023595909], datetime_start=datetime.datetime(2024, 4, 18, 10, 6, 11, 447018), datetime_complete=datetime.datetime(2024, 4, 18, 10, 6, 52, 91691), params={'subsample': 0.8646743646205938, 'learning_rate': 0.0884340928748416

In [94]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [95]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.758
train_ibs:  0.208


#### Test

In [96]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [97]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.0339977959383996,
                                 criterion='squared_error',
                                 dropout_rate=0.2075412325353082,
                                 learning_rate=0.010706280861824496,
                                 max_features='auto', max_leaf_nodes=17,
                                 min_impurity_decrease=3.3602815261835675e-07,
                                 min_samples_leaf=13, min_samples_split=18,
                                 min_weight_fraction_leaf=0.4472167339801619,
                                 n_estimators=445, random_state=123,
                                 subsample=0.9030031356045858,
                                 validation_fraction=0.9350158433232643)

C-index score: 0.618


GradientBoostingSurvivalAnalysis(ccp_alpha=0.008810810992982535,
                                 criterion='squared_error',
                                 dropout_rate=0.1693875032679634,
                                 learning_rate=0.08843409287484169,
                                 max_features='auto', max_leaf_nodes=15,
                                 min_impurity_decrease=5.157320437854681e-06,
                                 min_samples_leaf=16, min_samples_split=20,
                                 min_weight_fraction_leaf=0.29044798941528627,
                                 n_estimators=403, random_state=123,
                                 subsample=0.8646743646205938,
                                 validation_fraction=0.7489142536117352)

IBS: 0.213


In [98]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [99]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [100]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 10:25:03,330] A new study created in memory with name: no-name-6c6cfb35-8d81-4d3f-bafe-4a4e6d42ce76


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-18 10:25:12,048] Trial 0 finished with value: 0.6877154599339533 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6877154599339533.
Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-18 10:25:36,471] Trial 1 finished with value: 0.6877154599339533 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6877154599339533.
Fold 1 C-index: 0.5800865800865801
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7401960784313726
Fold 4 C-index: 0.6624472573839663
Fold 5 C-ind

Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.6877637130801688
Fold 5 C-index: 0.7323943661971831
[I 2024-04-18 10:30:31,232] Trial 19 finished with value: 0.7258622117301838 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7196046509277619, 'n_estimators': 431, 'learning_rate': 0.08300323114610605}. Best is trial 11 with value: 0.74027252284891.
Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.679324894514768
Fold 5 C-index: 0.7230046948356808
[I 2024-04-18 10:30:45,617] Trial 20 finished with value: 0.7130448585372656 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 258, 'learning_rate': 0.09795590448114745}. Best is trial 11 with value: 0.74027252284891.
Fold 1 C-index: 0.6363636363636364
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.8088235294117647
Fold

Fold 1 C-index: 0.5974025974025974
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7230046948356808
[I 2024-04-18 10:36:25,255] Trial 38 finished with value: 0.7004924785518998 and parameters: {'subsample': 0.6042961840380411, 'dropout_rate': 0.20387148741818267, 'n_estimators': 495, 'learning_rate': 0.09967365319698797}. Best is trial 11 with value: 0.74027252284891.
Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7276995305164319
[I 2024-04-18 10:36:45,975] Trial 39 finished with value: 0.7102206863555484 and parameters: {'subsample': 0.2832198393968297, 'dropout_rate': 0.161479375813903, 'n_estimators': 401, 'learning_rate': 0.07166694704505695}. Best is trial 11 with value: 0.74027252284891.
Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.662447257383966

...

Fold 2 IBS: 0.18223515335825835
Fold 3 IBS: 0.1777423595055709
Fold 4 IBS: 0.2018999578285449
Fold 5 IBS: 0.1911082943649028
[I 2024-04-18 11:06:32,632] Trial 19 finished with value: 0.19087084831171014 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.19087084831171014.
Fold 1 IBS: 0.19955188174503216
Fold 2 IBS: 0.1855544040643127
Fold 3 IBS: 0.17923234678472577
Fold 4 IBS: 0.20455539282706595
Fold 5 IBS: 0.19466473879647364
[I 2024-04-18 11:06:38,189] Trial 20 finished with value: 0.19271175284352204 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.19087084831171014.
Fold 1 IBS: 0.20256510757658133
Fold 2 IBS: 0.19154224065115247
Fold 3 IBS: 0.18335986646954017
Fold 4 IBS: 0.20458588921838688
Fold 5 IBS: 0.1974805846771631
[I 2024-04

Fold 2 IBS: 0.21852631898868966
Fold 3 IBS: 0.2030153396872452
Fold 4 IBS: 0.22215040490755078
Fold 5 IBS: 0.21594262691145144
[I 2024-04-18 11:09:00,021] Trial 38 finished with value: 0.21461165970615292 and parameters: {'subsample': 0.23557094034920928, 'dropout_rate': 0.13272164755980653, 'n_estimators': 1, 'learning_rate': 0.0947653216013069}. Best is trial 35 with value: 0.18994216705917563.
Fold 1 IBS: 0.2062176081503947
Fold 2 IBS: 0.2037315366365509
Fold 3 IBS: 0.1923016777876445
Fold 4 IBS: 0.2412731246694159
Fold 5 IBS: 0.2033237070159203
[I 2024-04-18 11:09:08,311] Trial 39 finished with value: 0.20936953085198526 and parameters: {'subsample': 0.167666800643059, 'dropout_rate': 0.19868896123028712, 'n_estimators': 111, 'learning_rate': 0.06892741183938003}. Best is trial 35 with value: 0.18994216705917563.
Fold 1 IBS: 0.20440240507823856
Fold 2 IBS: 0.1849206764977604
Fold 3 IBS: 0.18225096974199848
Fold 4 IBS: 0.21621756942600678
Fold 5 IBS: 0.18862470755011918
[I 2024-04-1

Fold 1 IBS: 0.2033663207076748
Fold 2 IBS: 0.18007792028319827
Fold 3 IBS: 0.17227783495267568
Fold 4 IBS: 0.20518748393382122
Fold 5 IBS: 0.18930167896409603
[I 2024-04-18 11:12:13,111] Trial 57 finished with value: 0.1900422477682932 and parameters: {'subsample': 0.101496956005944, 'dropout_rate': 0.49385760505702486, 'n_estimators': 215, 'learning_rate': 0.02025280377209624}. Best is trial 47 with value: 0.18608963039911053.
Fold 1 IBS: 0.20191437074126453
Fold 2 IBS: 0.18368497098361156
Fold 3 IBS: 0.17811956307175236
Fold 4 IBS: 0.2131306985901757
Fold 5 IBS: 0.18817122553751192
[I 2024-04-18 11:12:31,766] Trial 58 finished with value: 0.1930041657848632 and parameters: {'subsample': 0.23706967835523893, 'dropout_rate': 0.36573681723837637, 'n_estimators': 376, 'learning_rate': 0.006783089945917651}. Best is trial 47 with value: 0.18608963039911053.
Fold 1 IBS: 0.2022987991171443
Fold 2 IBS: 0.1859213832363397
Fold 3 IBS: 0.17823772867565482
Fold 4 IBS: 0.22102338598856885
Fold 5 

Fold 1 IBS: 0.21736310414126978
Fold 2 IBS: 0.22621598527429648
Fold 3 IBS: 0.2043056587143774
Fold 4 IBS: 0.2530734029856345
Fold 5 IBS: 0.20903981299264454
[I 2024-04-18 11:15:48,846] Trial 76 finished with value: 0.22199959282164455 and parameters: {'subsample': 0.19161322654962637, 'dropout_rate': 0.2274390090824162, 'n_estimators': 255, 'learning_rate': 0.030984968300376045}. Best is trial 66 with value: 0.18561301808622452.
Fold 1 IBS: 0.2093926746060527
Fold 2 IBS: 0.18747812089941904
Fold 3 IBS: 0.18350793543368743
Fold 4 IBS: 0.21000092056566644
Fold 5 IBS: 0.19094130798199171
[I 2024-04-18 11:15:57,900] Trial 77 finished with value: 0.1962641918973635 and parameters: {'subsample': 0.5891607666001992, 'dropout_rate': 0.17629353184919394, 'n_estimators': 138, 'learning_rate': 0.014071343257691903}. Best is trial 66 with value: 0.18561301808622452.
Fold 1 IBS: 0.19571917467377073
Fold 2 IBS: 0.18426023556786475
Fold 3 IBS: 0.1775312951028814
Fold 4 IBS: 0.19934669428156476
Fold 

Fold 1 IBS: 0.197838168025373
Fold 2 IBS: 0.1845307640679511
Fold 3 IBS: 0.17828065280806055
Fold 4 IBS: 0.20655436657739523
Fold 5 IBS: 0.19180023391350487
[I 2024-04-18 11:18:58,086] Trial 95 finished with value: 0.19180083707845694 and parameters: {'subsample': 0.17636450847859508, 'dropout_rate': 0.2683180256379605, 'n_estimators': 202, 'learning_rate': 0.009999123660042509}. Best is trial 66 with value: 0.18561301808622452.
Fold 1 IBS: 0.19796746552096606
Fold 2 IBS: 0.1828469400711107
Fold 3 IBS: 0.1762298250835142
Fold 4 IBS: 0.20469635327408078
Fold 5 IBS: 0.19168554513903255
[I 2024-04-18 11:19:07,879] Trial 96 finished with value: 0.19068522581774086 and parameters: {'subsample': 0.14665105862236183, 'dropout_rate': 0.3166812067394869, 'n_estimators': 147, 'learning_rate': 0.014533634685131247}. Best is trial 66 with value: 0.18561301808622452.
Fold 1 IBS: 0.20281841887854912
Fold 2 IBS: 0.1944235832270459
Fold 3 IBS: 0.19487494976385714
Fold 4 IBS: 0.2279662550105142
Fold 5 

In [101]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [102]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.746
train_ibs:  0.186


#### Test

In [103]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [104]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10443578020986953,
                                              learning_rate=0.09196255088220771,
                                              n_estimators=500,
                                              random_state=123,
                                              subsample=0.1072481243021174)

C-index score: 0.578


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.28132150572293957,
                                              learning_rate=0.02276408707314117,
                                              n_estimators=158,
                                              random_state=123,
                                              subsample=0.10041041298681407)

IBS: 0.234


In [105]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [106]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.886,1.0
Randomsurvivalforest,0.864,2.0
GradientBoosting,0.758,3.0
CoxElastic,0.753,4.0
ComponentwiseGradientBoosting,0.746,5.0
CoxRidge,0.716,6.0
CoxLasso,0.678,7.0


In [107]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.179,1.0
Randomsurvivalforest,0.183,2.0
ComponentwiseGradientBoosting,0.186,3.0
GradientBoosting,0.208,4.0
CoxElastic,0.210,5.0
CoxRidge,0.217,6.0
CoxLasso,0.313,7.0


In [108]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.662,1.0
Randomsurvivalforest,0.629,2.0
GradientBoosting,0.618,3.0
ComponentwiseGradientBoosting,0.578,4.0
CoxRidge,0.573,5.5
CoxElastic,0.573,5.5
CoxLasso,0.550,7.0


In [109]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
ExtraSurvivalTrees,0.200,1.0
Randomsurvivalforest,0.205,2.0
GradientBoosting,0.213,3.0
CoxRidge,0.221,4.5
CoxElastic,0.221,4.5
ComponentwiseGradientBoosting,0.234,6.0
CoxLasso,0.419,7.0


In [110]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/os/yeojohnson/no_selection/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_os_yeojohnson_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [111]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-18
